# Part 2 - Submitting Python Jobs on the SCC: CPU vs. GPU

Part 1 introduced submitting a basic batch job with `qsub`. This notebook covers the details you need for real workloads: **requesting the right resources**, the differences between **CPU** and **GPU** job scripts, and monitoring/managing jobs. This material is meant to be followed on the SCC itself.

## Anatomy of a Batch Script
Every SGE batch script is a shell script with `#$` directive lines. Common directives:

| Directive | Meaning |
|---|---|
| `-N name` | Job name |
| `-l h_rt=HH:MM:SS` | Maximum wall-clock run time |
| `-pe omp N` | Request N CPU cores (shared-memory parallel environment) |
| `-j y` | Merge stdout/stderr into one log file |
| `-m e` | Email when the job ends |
| `-l gpus=1` | Request 1 GPU |
| `-l gpu_c=X.X` | Minimum GPU compute capability |

Run `man qsub` or see the SCC documentation for the full list.

## Example 1: A CPU Job

`cpu_job.qsub`:
```bash
#!/bin/bash -l

#$ -N numpy_benchmark
#$ -l h_rt=00:30:00
#$ -pe omp 4          # Request 4 CPU cores
#$ -j y


module load miniconda
source activate energize

# Encourage NumPy's BLAS libraries to use the 4 allocated CPU cores.
export OMP_NUM_THREADS=4
export MKL_NUM_THREADS=4
export OPENBLAS_NUM_THREADS=4

python benchmark.py
```

Submit it from the directory containing `cpu_job.qsub` and `benchmark.py`:
```bash
qsub cpu_job.qsub
```

The thread variables encourage NumPy's underlying BLAS library to parallelize matrix multiplication across the four allocated cores. The actual behavior depends on the BLAS library installed in the environment.

## Example 2: A GPU Job

GPU jobs need two extra things: a GPU resource request, and (usually) a CUDA/GPU-enabled module.

`gpu_job.qsub`:
```bash
#!/bin/bash -l

#$ -N gpu_benchmark
#$ -l h_rt=00:30:00
#$ -l gpus=1                 # Request 1 GPU
#$ -l gpu_c=7.0               # Minimum compute capability
#$ -j y

module load miniconda
module load cuda/12.8
conda activate energize

# Use the C++ runtime bundled with the conda environment.
export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:${LD_LIBRARY_PATH:-}"

python gpu_benchmark.py
```

Submit the same way:
```bash
qsub gpu_job.qsub
```

Use `qgpus` to see what GPU models are currently available on the cluster, and their current usage.

## Monitoring and Managing Jobs

| Command | Purpose |
|---|---|
| `qstat -u your_username` | List your queued/running jobs |
| `qstat -j <job_id>` | Detailed status of a specific job |
| `qdel <job_id>` | Cancel a job |
| `qacct -j <job_id>` | Resource usage report for a finished job (CPU time, memory, wall time) |

Checking `qacct` after a job finishes is a good habit - it tells you whether you requested too much (wasting your allocation) or too little (risking your job being killed for exceeding its time/memory limit) for next time.